In [2]:
# Cell 0 — setup, detect project root & processed folder
from pathlib import Path
import pandas as pd
import warnings, shutil, difflib
warnings.simplefilter(action='ignore', category=FutureWarning)

CWD = Path.cwd()
# if running from notebooks/ parent likely project root
if (CWD / ".." / "data").exists():
    PROJECT_ROOT = (CWD / "..").resolve()
else:
    PROJECT_ROOT = CWD.resolve()

PROC = PROJECT_ROOT / "data" / "processed"
RAW  = PROJECT_ROOT / "data" / "raw"
PROC.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Processed folder:", PROC)
print("Files in processed:", [p.name for p in sorted(PROC.glob("*.csv"))])


PROJECT_ROOT: C:\Users\Ali Sherif\Apple-Retail-Sales-Forcasting
Processed folder: C:\Users\Ali Sherif\Apple-Retail-Sales-Forcasting\data\processed
Files in processed: ['cleaned_apple_sales.csv', 'cleaned_apple_sales_AUTO_MAP_BACKUP.csv', 'cleaned_apple_sales_auto_mapped.csv', 'cleaned_apple_sales_BACKUP.csv', 'cleaned_apple_sales_BEFORE_ENRICH_BACKUP.csv', 'cleaned_apple_sales_enriched.csv', 'cleaned_apple_sales_enriched_auto.csv', 'cleaned_exchange.csv', 'cleaned_gdp.csv', 'cleaned_inflation.csv', 'cleaned_internet.csv', 'merged_apple_enriched.csv', 'merged_apple_enriched_auto.csv']


In [3]:
# Cell 1 — ensure Taiwan rows exist in externals (idempotent)
PROC = PROC  # from Cell 0

taiwan_values = {
    "gdp": [
        ("taiwan", 2020, 676900000000.0),
        ("taiwan", 2021, 776900000000.0),
        ("taiwan", 2022, 765600000000.0),
        ("taiwan", 2023, 757300000000.0),
        ("taiwan", 2024, 797000000000.0),
    ],
    "exchange_rate": [
        ("taiwan", 2020, 29.5),
        ("taiwan", 2021, 28.0),
        ("taiwan", 2022, 29.5),
        ("taiwan", 2023, 30.8),
        ("taiwan", 2024, 31.5),
    ],
    "inflation_rate": [
        ("taiwan", 2020, -0.2),
        ("taiwan", 2021, 1.8),
        ("taiwan", 2022, 2.9),
        ("taiwan", 2023, 2.5),
        ("taiwan", 2024, 2.2),
    ],
    "internet_usage_pct": [
        ("taiwan", 2020, 90.0),
        ("taiwan", 2021, 91.0),
        ("taiwan", 2022, 92.0),
        ("taiwan", 2023, 93.0),
        ("taiwan", 2024, 94.0),
    ]
}

def append_if_missing(file, colname, rows):
    if not (PROC / file).exists():
        print("File missing, skipping:", file)
        return
    df = pd.read_csv(PROC / file, low_memory=False)
    to_add = []
    for country, year, val in rows:
        mask = (df['country_norm'].astype(str).str.lower() == country) & (df['year'] == int(year))
        if not mask.any():
            to_add.append({ "country_norm": country, "year": int(year), colname: val })
    if to_add:
        df2 = pd.concat([df, pd.DataFrame(to_add)], ignore_index=True)
        df2['year'] = df2['year'].astype(int)
        (PROC / file).rename(PROC / (file + ".bak"), exist_ok=False) if False else None
        df2.to_csv(PROC / file, index=False)
        print(f"Appended {len(to_add)} rows to {file}")
    else:
        print(f"No append needed for {file}")

# run for each macro file
append_if_missing("cleaned_gdp.csv", "gdp", taiwan_values["gdp"])
append_if_missing("cleaned_exchange.csv", "exchange_rate", taiwan_values["exchange_rate"])
append_if_missing("cleaned_inflation.csv", "inflation_rate", taiwan_values["inflation_rate"])
append_if_missing("cleaned_internet.csv", "internet_usage_pct", taiwan_values["internet_usage_pct"])


No append needed for cleaned_gdp.csv
No append needed for cleaned_exchange.csv
No append needed for cleaned_inflation.csv
No append needed for cleaned_internet.csv


In [4]:
# Cell 2 — ensure 'korea, rep.' 2024 GDP is present and filled (idempotent)
gdp_file = PROC / "cleaned_gdp.csv"
if not gdp_file.exists():
    raise FileNotFoundError("cleaned_gdp.csv not found in processed. Run preprocessing first.")

gdf = pd.read_csv(gdp_file, low_memory=False)

mask_k2024 = (gdf['country_norm'].astype(str).str.lower() == "korea, rep.") & (gdf['year'] == 2024)
if mask_k2024.any():
    # if row exists but value is NaN, fill it
    idx = gdf.loc[mask_k2024].index
    if pd.isna(gdf.loc[idx, 'gdp']).all():
        gdf.loc[idx, 'gdp'] = 1869700000000.0  # 1.8697e12 USD
        gdf.to_csv(gdp_file, index=False)
        print("Filled Korea 2024 GDP value.")
    else:
        print("Korea 2024 GDP row exists and is non-empty.")
else:
    # append the row
    new = pd.DataFrame([{"country_norm": "korea, rep.", "year": 2024, "gdp": 1869700000000.0}])
    gdf2 = pd.concat([gdf, new], ignore_index=True)
    gdf2['year'] = gdf2['year'].astype(int)
    gdf2.to_csv(gdp_file, index=False)
    print("Appended Korea 2024 GDP row.")


Korea 2024 GDP row exists and is non-empty.


In [5]:
# Cell 3 — load sales and external cleaned macro files and ensure keys exist
sales_path = PROC / "cleaned_apple_sales.csv"
if not sales_path.exists():
    raise FileNotFoundError("cleaned_apple_sales.csv not found — run sales cleaning notebook first (01).")

sales = pd.read_csv(sales_path, low_memory=False, parse_dates=["sale_date"])
print("Loaded sales shape:", sales.shape)

# ensure year and normalized country
if 'year' not in sales.columns:
    sales['year'] = sales['sale_date'].dt.year.astype(int)

sales['country_norm'] = sales['country'].astype(str).str.strip().str.lower().str.replace(r"\s+"," ", regex=True)
if 'country_norm_mapped' not in sales.columns:
    sales['country_norm_mapped'] = sales['country_norm']

# load externals
gdp       = pd.read_csv(PROC / "cleaned_gdp.csv", low_memory=False)
exchange  = pd.read_csv(PROC / "cleaned_exchange.csv", low_memory=False)
inflation = pd.read_csv(PROC / "cleaned_inflation.csv", low_memory=False)
internet  = pd.read_csv(PROC / "cleaned_internet.csv", low_memory=False)

print("Externals shapes => gdp, exchange, inflation, internet:", len(gdp), len(exchange), len(inflation), len(internet))


Loaded sales shape: (1040200, 16)
Externals shapes => gdp, exchange, inflation, internet: 17295 17295 17295 17295


In [6]:
# Cell 4 — apply manual country mapping and apply conservative auto-mapping
manual_country_map = {
    "uae": "united arab emirates",
    "united arab emirates of": "united arab emirates",
    "south korea": "korea, rep.",
    "korea south": "korea, rep.",
    "usa": "united states",
    "u.s.a.": "united states",
    "uk": "united kingdom",
    "england": "united kingdom",
    "ivory coast": "cote d'ivoire",
    "russia": "russian federation",
}

# apply manual mapping
sales['country_norm_mapped'] = sales['country_norm'].replace(manual_country_map)

# conservative auto-mapping (only very close difflib matches)
external_countries = sorted(gdp['country_norm'].dropna().unique())

def safe_automap(sales_series, external_choices, cutoff=0.85):
    auto_map = {}
    for s in sorted(sales_series.dropna().unique()):
        if s in external_choices: 
            continue
        if s in manual_country_map:
            continue
        match = difflib.get_close_matches(s, external_choices, n=1, cutoff=cutoff)
        if match:
            auto_map[s] = match[0]
    return auto_map

auto_map = safe_automap(sales['country_norm_mapped'], external_countries, cutoff=0.85)
print("Conservative auto-map found:", auto_map)
sales['country_norm_mapped'] = sales['country_norm_mapped'].replace(auto_map)

# show remaining unmatched sample
remaining = sorted(set(sales['country_norm_mapped'].dropna().unique()) - set(external_countries))
print("Unmatched sales country names (sample):", remaining[:30])


Conservative auto-map found: {}
Unmatched sales country names (sample): []


In [7]:
# Cell 5 — build lookups keyed by 'country_norm|year'
def make_lookup(df, value_col):
    df2 = df.copy()
    df2['__k'] = df2['country_norm'].astype(str) + '|' + df2['year'].astype(str)
    df2 = df2.drop_duplicates('__k', keep='last')
    return df2.set_index('__k')[value_col]

gdp_lookup = make_lookup(gdp, 'gdp')
exchange_lookup = make_lookup(exchange, 'exchange_rate')
infl_lookup = make_lookup(inflation, 'inflation_rate')
internet_lookup = make_lookup(internet, 'internet_usage_pct')

print("Lookups sizes:", len(gdp_lookup), len(exchange_lookup), len(infl_lookup), len(internet_lookup))


Lookups sizes: 17295 17295 17295 17295


In [8]:
# Cell 6 — map macro values into sales using sales.country_norm_mapped + year
sales['__k'] = sales['country_norm_mapped'].astype(str) + '|' + sales['year'].astype(str)

sales['gdp'] = sales['__k'].map(gdp_lookup)
sales['exchange_rate'] = sales['__k'].map(exchange_lookup)
sales['inflation_rate'] = sales['__k'].map(infl_lookup)
sales['internet_usage_pct'] = sales['__k'].map(internet_lookup)

# drop helper key
sales.drop(columns=['__k'], inplace=True)

print("After mapping - missing counts:")
for c in ['gdp','exchange_rate','inflation_rate','internet_usage_pct']:
    print(c, int(sales[c].isna().sum()))


After mapping - missing counts:
gdp 0
exchange_rate 0
inflation_rate 0
internet_usage_pct 106716


In [9]:
# Cell 7 — save merged_apple_enriched.csv (macro-only)
out_merged = PROC / "merged_apple_enriched.csv"
cols_to_keep = ['sale_id','sale_date','country','year','country_norm','country_norm_mapped']
for c in ['gdp','exchange_rate','inflation_rate','internet_usage_pct']:
    if c in sales.columns and c not in cols_to_keep:
        cols_to_keep.append(c)

sales[cols_to_keep].to_csv(out_merged, index=False)
print("Saved merged macro file to:", out_merged)
print("Merged shape:", sales[cols_to_keep].shape)


Saved merged macro file to: C:\Users\Ali Sherif\Apple-Retail-Sales-Forcasting\data\processed\merged_apple_enriched.csv
Merged shape: (1040200, 10)


In [10]:
# Cell 8 — merge into final cleaned_apple_sales_enriched.csv (backup original)
clean_file = PROC / "cleaned_apple_sales.csv"
if not clean_file.exists():
    raise FileNotFoundError("cleaned_apple_sales.csv missing; run sales cleaning step first")

# backup original cleaned sales
backup = PROC / "cleaned_apple_sales_BEFORE_ENRICH_BACKUP.csv"
if not backup.exists():
    shutil.copy2(clean_file, backup)
    print("Backup created:", backup)

clean = pd.read_csv(clean_file, low_memory=False, parse_dates=['sale_date'])
merged = pd.read_csv(out_merged, low_memory=False, parse_dates=['sale_date'])

macro_cols = [c for c in ['gdp','exchange_rate','inflation_rate','internet_usage_pct','country_norm','country_norm_mapped','year'] if c in merged.columns and c not in clean.columns]
print("Macro cols to add:", macro_cols)

merged_sub = merged[['sale_id'] + macro_cols]
combined = clean.merge(merged_sub, how='left', on='sale_id')

out_final = PROC / "cleaned_apple_sales_enriched.csv"
combined.to_csv(out_final, index=False)
print("Final enriched saved to:", out_final)
print("Final shape:", combined.shape)


Macro cols to add: ['gdp', 'exchange_rate', 'inflation_rate', 'internet_usage_pct', 'country_norm', 'country_norm_mapped', 'year']
Final enriched saved to: C:\Users\Ali Sherif\Apple-Retail-Sales-Forcasting\data\processed\cleaned_apple_sales_enriched.csv
Final shape: (1040200, 23)


In [11]:
# Cell 9 — diagnostics & show problematic countries
final = combined

print("\n=== Overall missing counts ===")
for c in ['gdp','exchange_rate','inflation_rate','internet_usage_pct']:
    print(c, int(final[c].isna().sum()))

print("\n=== Missing by Year (gdp/exchange/inflation) ===")
print(final.groupby('year')[[c for c in ['gdp','exchange_rate','inflation_rate'] if c in final.columns]].apply(lambda x: x.isna().sum()))

print("\n=== Missing by country (top problematic) ===")
grp = final.groupby('country_norm_mapped').agg(total_rows=('sale_id','size'), missing_gdp=('gdp', lambda s: s.isna().sum()), missing_internet=('internet_usage_pct', lambda s: s.isna().sum()))
grp['miss_frac_gdp'] = grp['missing_gdp'] / grp['total_rows']
display(grp.sort_values('missing_gdp', ascending=False).head(40))

print("\nIf you want me to add more manual mappings, tell me the exact sales name -> external country_norm and I'll update the mapping section of the notebook.")



=== Overall missing counts ===
gdp 0
exchange_rate 0
inflation_rate 0
internet_usage_pct 106716

=== Missing by Year (gdp/exchange/inflation) ===
      gdp  exchange_rate  inflation_rate
year                                    
2020    0              0               0
2021    0              0               0
2022    0              0               0
2023    0              0               0
2024    0              0               0

=== Missing by country (top problematic) ===


,total_rows,missing_gdp,missing_internet,miss_frac_gdp
country_norm_mapped,,,,
australia,97280,0,17461,0.0
austria,13771,0,0,0.0
canada,69468,0,12257,0.0
china,97022,0,0,0.0
colombia,27524,0,4965,0.0
france,55119,0,0,0.0
germany,41977,0,0,0.0
italy,27665,0,0,0.0
japan,83697,0,15132,0.0



If you want me to add more manual mappings, tell me the exact sales name -> external country_norm and I'll update the mapping section of the notebook.


In [12]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
PROC = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(PROC / "cleaned_apple_sales_enriched.csv")
df.columns = [col.lower() for col in df.columns]
df.columns

Index(['sale_id', 'sale_date', 'store_id', 'product_id', 'quantity',
       'product_name', 'launch_date', 'price', 'store_name', 'city', 'country',
       'category_id', 'category_name', 'sales_amount', 'invalid_launch_flag',
       'product_age_days', 'gdp', 'exchange_rate', 'inflation_rate',
       'internet_usage_pct', 'country_norm', 'country_norm_mapped', 'year'],
      dtype='object')

In [14]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
PROC = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(PROC / "cleaned_apple_sales_enriched.csv")
df['country_norm'] = df['country_norm'].astype(str).str.lower().str.strip()

countries = ["taiwan", "south korea"]  # <- these are the actual names inside your enriched file

for c in countries:
    sub = df[df['country_norm'] == c]
    print(f"\nCountry: {c} | Rows: {len(sub)}")

    print("Missing GDP:", sub['gdp'].isna().sum())
    print("Missing Inflation:", sub['inflation_rate'].isna().sum())
    print("Missing Exchange:", sub['exchange_rate'].isna().sum())

    # Show if there are any problematic rows
    bad = sub[sub[['gdp','inflation_rate','exchange_rate']].isna().any(axis=1)]
    print("Problematic rows:", len(bad))



Country: taiwan | Rows: 13982
Missing GDP: 0
Missing Inflation: 0
Missing Exchange: 0
Problematic rows: 0

Country: south korea | Rows: 27912
Missing GDP: 0
Missing Inflation: 0
Missing Exchange: 0
Problematic rows: 0


In [17]:
df.head()

,sale_id,sale_date,store_id,product_id,quantity,product_name,launch_date,price,store_name,city,...,sales_amount,invalid_launch_flag,product_age_days,gdp,exchange_rate,inflation_rate,internet_usage_pct,country_norm,country_norm_mapped,year
0,RW-18212,2020-01-01,ST-41,P-73,2,Apple One,2020-01-01,488,Apple Central World,Bangkok,...,976,True,0,6985.643939,31.293673,-0.845937,77.8437,thailand,thailand,2020
1,RX-6848,2020-01-01,ST-31,P-40,7,iPhone SE (3rd Generation),2020-01-01,923,Apple Shinjuku,Tokyo,...,6461,True,0,40028.734170,106.774582,-0.024996,90.2195,japan,japan,2020
2,SV-39198,2020-01-01,ST-9,P-73,7,Apple One,2020-01-01,488,Apple Park Visitor Center,Cupertino,...,3416,True,0,64401.507440,1.000000,1.233584,90.3447,united states,united states,2020
3,ZI-218670,2020-01-01,ST-20,P-70,1,Apple Fitness+,2020-01-01,923,Apple Kaerntner Strasse,Vienna,...,923,True,0,48716.409890,0.875506,1.381911,87.5294,austria,austria,2020
4,VB-48589,2020-01-01,ST-60,P-18,9,Beats Solo Pro,2020-01-01,1773,Apple Antara,Mexico City,...,15957,True,0,8841.270751,21.485608,3.396834,71.4902,mexico,mexico,2020
